<div style="background: linear-gradient(135deg, #1a1a2e 0%, #16213e 50%, #0f3460 100%);
            padding: 48px 36px; border-radius: 14px; text-align: center;
            color: white; margin-bottom: 6px; font-family: Arial, sans-serif;">



  <h1 style="font-size: 2.5em; margin: 0 0 10px 0; letter-spacing: 1px; font-weight: 700;">
    🦠 COVID-19 Global Pandemic Analysis
  </h1>

  <p style="font-size: 1.15em; opacity: 0.85; margin: 0 0 20px 0; font-weight: 300;">
    Data Cleaning & Preparation for Exploratory Analysis 

  </p>

  <hr style="border: none; border-top: 1px solid rgba(255,255,255,0.2); margin: 20px auto; width: 60%;">

  <div style="display: flex; justify-content: center; gap: 40px; flex-wrap: wrap;
              font-size: 0.88em; opacity: 0.75; margin-top: 6px;">
    <span>📊  Interactive Charts</span>
    <span>🐍 Python · Plotly · Pandas</span>
  </div>
</div>


**Dataset:** Our World in Data (OWID) COVID-19 Dataset  
**Source:** [Our World in Data — COVID-19](https://github.com/owid/covid-19-data)  
**Project type:** Data Cleaning & Preparation for Exploratory Analysis  

---

| Field | Detail |
|---|---|
| Data granularity | Country × Day (longitudinal panel) |
| Date coverage | January 2020 – present |
| Variables | 67 columns across 8 thematic groups |
| Upstream sources | WHO, Johns Hopkins, Oxford OxCGRT, national health ministries |

---

> **Navigation guide:** Sections 1–3 document the dataset and cleaning rationale — read these before running any code. Sections 4–7 load, inspect, and assess the raw data. Section 8 applies all cleaning transformations by variable group. Sections 9–11 validate results, surface remaining limitations, and export the final dataset.


## Abstract

This notebook implements a structured data cleaning pipeline for the Our World in Data (OWID) COVID-19 dataset. The raw file contains 67 columns mixing variables of fundamentally different types: cumulative counters, daily flows, derived smoothed series, bounded rates, and static country-level characteristics. Each type requires a distinct treatment.

The most common error in published analyses of this dataset is applying a single imputation rule across all columns — typically forward-fill or zero-fill — regardless of variable type. This silently corrupts multiple series. For example, back-filling a 7-day smoothed average introduces future data into past observations (leakage). Filling a missing per-million rate with zero asserts that cases were zero, not that data was absent.

Every cleaning decision in this notebook is explained alongside the code: what strategy was chosen, why it fits this variable type, and what the alternatives would have produced. The output is a single CSV file ready for exploratory analysis or modeling.


## 1. Research Context

The OWID COVID-19 dataset is a longitudinal panel in which each row represents one country on one calendar day. It draws from multiple upstream sources — the WHO, Johns Hopkins University, national health ministries, the Oxford Government Response Tracker (OxCGRT), and others — meaning that coverage, update frequency, and reporting conventions differ across columns and across countries.

Cleaning this dataset is not a mechanical exercise. The dataset mixes four structurally different variable types:

- **Cumulative counters** — values that should only increase over time (total cases, total deaths, total vaccinations). Gaps arise when a country misses a reporting day; the previous total is still valid.
- **Daily flows** — values that measure change within a single day (new cases, new deaths, new tests). A missing day is genuinely unknown — not zero. Negative values arise from retroactive corrections and must be treated as errors.
- **Derived smoothed series** — 7-day rolling averages computed from raw flows (new_cases_smoothed, new_deaths_smoothed). These are calculated by OWID from the raw data. Filling them independently, or back-filling them, introduces inconsistencies with the source series.
- **Static country indicators** — fixed values that do not change over time (GDP per capita, median age, life expectancy). They repeat identically across every row for a given country and should propagate in both directions within that country.

**Primary cleaning objectives:**
1. Assign correct data types to all columns
2. Apply a strategy matched to each variable type — not a uniform rule
3. Preserve structural nulls rather than fabricating values
4. Correct genuine errors (negative counts, out-of-range rates) without distorting real signal
5. Produce a clean dataset with documented residual limitations


## 2. Dataset Overview

The OWID dataset covers most countries from early 2020 onward. Coverage is not uniform: high-income countries with established public health surveillance contribute dense, daily data across most columns. Many low-income countries have partial or intermittent reporting, particularly for testing and hospital capacity.

The dataset also includes OWID-constructed aggregate rows — entries for "World," "Europe," regional income groups, and similar groupings — identified by an `iso_code` starting with `OWID_`. These are useful for global-level analyses but must be excluded before any country-level cleaning, since groupby operations over `location` would mix country rows with aggregate rows.

**Column groups:**

| Group | Key columns |
|---|---|
| Identifiers | `iso_code`, `continent`, `location`, `date` |
| Confirmed cases | `total_cases`, `new_cases`, `new_cases_smoothed`, per-million variants |
| Confirmed deaths | `total_deaths`, `new_deaths`, `new_deaths_smoothed`, per-million variants |
| Excess mortality | `excess_mortality`, cumulative and per-million variants |
| Hospital & ICU | `icu_patients`, `hosp_patients`, weekly admissions, per-million variants |
| Policy responses | `stringency_index` |
| Reproduction rate | `reproduction_rate` |
| Testing | `total_tests`, `new_tests`, `positive_rate`, `tests_per_case`, `tests_units` |
| Vaccinations | `total_vaccinations`, `people_vaccinated`, `people_fully_vaccinated`, `total_boosters`, smoothed and per-hundred variants |
| Country indicators | population, density, demographics, economic indicators, health system capacity |


## 3. Data Dictionary — Cleaning Rationale by Variable Type

The cleaning strategy for each column follows directly from what the variable measures. The table below maps variable types to their expected behavior and the appropriate cleaning approach. Applying the wrong strategy to a variable type is a source of analytical error, not just a style choice.

| Variable type | Expected behavior | Cleaning approach |
|---|---|---|
| Cumulative counter | Monotonically non-decreasing within a country | `ffill()` within country to fill report gaps, then `cummax()` to correct retroactive downward revisions |
| Daily flow | Can be zero; should not be negative | Null out negatives; leave gaps as `NaN` — do not substitute zero for missing |
| Smoothed / derived | Computed from raw series by OWID; lags the signal | Preserve nulls; do not back-fill (introduces future data into earlier rows) |
| Per-unit normalized | Derived from raw ÷ population | Do not recompute unless necessary; do not fill with zero; null out negatives |
| Rate / proportion | Bounded [0, 1] or [0, 100] | Clip to valid range; forward-fill with a limited window |
| Static indicator | One fixed value per country, repeated on every row | `ffill()` then `bfill()` within country only |
| Categorical string | Fixed vocabulary | Normalize casing; `ffill()` within country; encode as `category` |

**Known structural coverage gaps (not errors):**

- Excess mortality data exists for approximately 60 countries — those with reliable civil registration systems. Nulls elsewhere are not imputable.
- Hospital and ICU data was primarily reported by European and high-income countries. Sparse global coverage is expected.
- Testing data improved over time but remains incomplete for many low-income countries, particularly in sub-Saharan Africa.
- Vaccination data begins in late 2020 and has variable completeness in later years as some countries stopped reporting.


## 4. Import Libraries


In [55]:
import pandas as pd
import numpy as np
import warnings

warnings.filterwarnings('ignore')

# Show all columns when printing DataFrames
pd.set_option('display.max_columns', None)

# Consistent float display (4 decimal places with thousand separators)
pd.set_option('display.float_format', lambda x: f'{x:,.4f}')


## 5. Load Data

Set `FILE_PATH` to the location of your local copy of the OWID dataset. The raw CSV can be downloaded from the [OWID GitHub repository](https://github.com/owid/covid-19-data/tree/master/public/data).

The raw DataFrame (`df_raw`) is preserved throughout. All transformations operate on `df`, a working copy. This ensures the original data remains accessible for comparison or auditing at any point.


In [56]:
# Update FILE_PATH to point to your local copy of owid-covid-data.csv
FILE_PATH = r'F:\faculty\Level 3 S_2\Data Visualization\Global COVID-19 Pandemic Analysis\Global-COVID-19-Pandemic-Analysis\Data\Raw\owid-covid-data.csv'

df_raw = pd.read_csv(FILE_PATH, low_memory=False)
df = df_raw.copy()   # preserve the raw source; all edits go to df

print(f"Rows: {df.shape[0]:,}    Columns: {df.shape[1]}")


Rows: 429,435    Columns: 67


## 6. Initial Data Inspection

The three cells below give a structural overview of the raw dataset before any modifications: a sample of rows, column metadata (types and non-null counts), and summary statistics across all columns. Read these outputs before proceeding — they establish the baseline from which the cleaning assessment is built.


In [57]:
df.head(3)


,iso_code,continent,location,date,total_cases,new_cases,new_cases_smoothed,total_deaths,new_deaths,new_deaths_smoothed,total_cases_per_million,new_cases_per_million,new_cases_smoothed_per_million,total_deaths_per_million,new_deaths_per_million,new_deaths_smoothed_per_million,reproduction_rate,icu_patients,icu_patients_per_million,hosp_patients,hosp_patients_per_million,weekly_icu_admissions,weekly_icu_admissions_per_million,weekly_hosp_admissions,weekly_hosp_admissions_per_million,total_tests,new_tests,total_tests_per_thousand,new_tests_per_thousand,new_tests_smoothed,new_tests_smoothed_per_thousand,positive_rate,tests_per_case,tests_units,total_vaccinations,people_vaccinated,people_fully_vaccinated,total_boosters,new_vaccinations,new_vaccinations_smoothed,total_vaccinations_per_hundred,people_vaccinated_per_hundred,people_fully_vaccinated_per_hundred,total_boosters_per_hundred,new_vaccinations_smoothed_per_million,new_people_vaccinated_smoothed,new_people_vaccinated_smoothed_per_hundred,stringency_index,population_density,median_age,aged_65_older,aged_70_older,gdp_per_capita,extreme_poverty,cardiovasc_death_rate,diabetes_prevalence,female_smokers,male_smokers,handwashing_facilities,hospital_beds_per_thousand,life_expectancy,human_development_index,population,excess_mortality_cumulative_absolute,excess_mortality_cumulative,excess_mortality,excess_mortality_cumulative_per_million
0,AFG,Asia,Afghanistan,2020-01-05,0.0000,0.0000,NaN,0.0000,0.0000,NaN,0.0000,0.0000,NaN,0.0000,0.0000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0000,54.4200,18.6000,2.5800,1.3400,"1,803.9900",NaN,597.0300,9.5900,NaN,NaN,37.7500,0.5000,64.8300,0.5100,41128772,NaN,NaN,NaN,NaN
1,AFG,Asia,Afghanistan,2020-01-06,0.0000,0.0000,NaN,0.0000,0.0000,NaN,0.0000,0.0000,NaN,0.0000,0.0000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0000,54.4200,18.6000,2.5800,1.3400,"1,803.9900",NaN,597.0300,9.5900,NaN,NaN,37.7500,0.5000,64.8300,0.5100,41128772,NaN,NaN,NaN,NaN
2,AFG,Asia,Afghanistan,2020-01-07,0.0000,0.0000,NaN,0.0000,0.0000,NaN,0.0000,0.0000,NaN,0.0000,0.0000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0000,54.4200,18.6000,2.5800,1.3400,"1,803.9900",NaN,597.0300,9.5900,NaN,NaN,37.7500,0.5000,64.8300,0.5100,41128772,NaN,NaN,NaN,NaN


In [58]:
df.info(verbose=True, show_counts=True)


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 429435 entries, 0 to 429434
Data columns (total 67 columns):
 #   Column                                      Non-Null Count   Dtype  
---  ------                                      --------------   -----  
 0   iso_code                                    429435 non-null  object 
 1   continent                                   402910 non-null  object 
 2   location                                    429435 non-null  object 
 3   date                                        429435 non-null  object 
 4   total_cases                                 411804 non-null  float64
 5   new_cases                                   410159 non-null  float64
 6   new_cases_smoothed                          408929 non-null  float64
 7   total_deaths                                411804 non-null  float64
 8   new_deaths                                  410608 non-null  float64
 9   new_deaths_smoothed                         409378 non-null  float64
 

In [59]:
df.describe(include='all').T


,count,unique,top,freq,mean,std,min,25%,50%,75%,max
iso_code,429435,255,OWID_HIC,3026,NaN,NaN,NaN,NaN,NaN,NaN,NaN
continent,402910,6,Africa,95419,NaN,NaN,NaN,NaN,NaN,NaN,NaN
location,429435,255,High-income countries,3026,NaN,NaN,NaN,NaN,NaN,NaN,NaN
date,429435,1688,2022-01-10,261,NaN,NaN,NaN,NaN,NaN,NaN,NaN
total_cases,"411,804.0000",NaN,NaN,NaN,"7,365,292.3545","44,775,816.7667",0.0000,"6,280.7500","63,653.0000","758,272.0000","775,866,783.0000"
...,...,...,...,...,...,...,...,...,...,...,...
population,"429,435.0000",NaN,NaN,NaN,"152,033,640.3963","697,540,771.6681",47.0000,"523,798.0000","6,336,393.0000","32,969,520.0000","7,975,105,024.0000"
excess_mortality_cumulative_absolute,"13,411.0000",NaN,NaN,NaN,"56,047.6536","156,869.0756","-37,726.1000",176.5000,"6,815.2000","39,128.0450","1,349,776.4000"
excess_mortality_cumulative,"13,411.0000",NaN,NaN,NaN,9.7664,12.0407,-44.2300,2.0600,8.1300,15.1600,78.0800
excess_mortality,"13,411.0000",NaN,NaN,NaN,10.9254,24.5607,-95.9200,-1.5000,5.6600,15.5750,378.2200


## 7. Data Quality Assessment

This section documents the raw dataset's quality issues before any cleaning takes place: missing value rates by column, duplicate row checks, OWID aggregate rows that need to be excluded, and date range coverage. Running this assessment first ensures that every subsequent cleaning decision is grounded in observed data, not assumptions.


In [60]:
# Missing value rates — sorted descending so the most incomplete columns appear first
missing = df.isnull().mean().mul(100).round(2).sort_values(ascending=False)
missing_df = missing[missing > 0].reset_index()
missing_df.columns = ['column', 'pct_missing']
print(missing_df.to_string(index=False))


                                    column  pct_missing
         weekly_icu_admissions_per_million      97.4400
                     weekly_icu_admissions      97.4400
   excess_mortality_cumulative_per_million      96.8800
                          excess_mortality      96.8800
               excess_mortality_cumulative      96.8800
      excess_mortality_cumulative_absolute      96.8800
        weekly_hosp_admissions_per_million      94.3000
                    weekly_hosp_admissions      94.3000
                              icu_patients      90.8900
                  icu_patients_per_million      90.8900
                 hosp_patients_per_million      90.5300
                             hosp_patients      90.5300
                total_boosters_per_hundred      87.5200
                            total_boosters      87.5200
                          new_vaccinations      83.4700
                                 new_tests      82.4400
                    new_tests_per_thousand      

In [61]:
# Check for duplicate (location, date) pairs — should be zero in a well-formed panel
n_dupes = df.duplicated(subset=['location', 'date']).sum()
print(f"Duplicate (location, date) pairs: {n_dupes}")


Duplicate (location, date) pairs: 7770


In [62]:
# Identify OWID-constructed aggregate rows (continents, income groups, etc.)
# These share iso_code prefixes like 'OWID_EUR', 'OWID_LIC', 'OWID_WRL'
# They must be removed before country-level groupby operations
owid_entries = df[df['iso_code'].str.startswith('OWID', na=False)]['location'].unique()
print(f"OWID aggregates identified for removal ({len(owid_entries)} entries):")
print(owid_entries)


OWID aggregates identified for removal (18 entries):
['Africa' 'Asia' 'England' 'Europe' 'European Union (27)'
 'High-income countries' 'Kosovo' 'Low-income countries'
 'Lower-middle-income countries' 'North America' 'Northern Cyprus'
 'Northern Ireland' 'Oceania' 'Scotland' 'South America'
 'Upper-middle-income countries' 'Wales' 'World']


In [63]:
# Date range and country count in the raw dataset (before filtering)
print(f"Date range : {df['date'].min()}  →  {df['date'].max()}")
print(f"Unique locations (including aggregates): {df['location'].nunique()}")


Date range : 2020-01-01  →  2024-08-14
Unique locations (including aggregates): 255


## 8. Data Cleaning

Cleaning is organized by column group, following the logic established in Section 3. Each subsection opens with the rationale for the chosen strategy and any deviations from naive approaches. Reading the rationale is part of using this notebook correctly — the code alone does not capture the analytical decisions embedded in it.

OWID aggregate rows are removed at the start of Section 8.1 so that all subsequent `groupby('location')` operations work on individual countries only.


### 8.1 Identifiers & Metadata

The four identifier columns — `iso_code`, `continent`, `location`, and `date` — form the index of the panel. String identifiers are stripped of whitespace and normalized to title case. The `date` column is parsed from string to `datetime64`. OWID regional aggregate rows are removed at this step.

`iso_code` is dropped after filtering: downstream analysis uses `location` (country name) as the primary identifier, and `iso_code` is redundant once aggregates are excluded.

The DataFrame is sorted chronologically within each country to ensure that `ffill()` and `cummax()` operations proceed in the correct temporal direction throughout the rest of this notebook.


In [64]:
# Normalize string identifiers
df['iso_code']  = df['iso_code'].astype(str).str.strip()
df['continent'] = df['continent'].astype(str).str.strip().str.title()
df['location']  = df['location'].astype(str).str.strip()

# Parse date strings to proper datetime objects
df['date'] = pd.to_datetime(df['date'], errors='coerce')

# Remove OWID regional aggregates — keep only rows with standard ISO country codes
df = df[~df['iso_code'].str.startswith('OWID', na=False)].copy()

# Drop iso_code — location serves as the country identifier from here on
df.drop('iso_code', axis=1, inplace=True)

# Sort chronologically within each country (required for ffill and cummax to be meaningful)
df = df.sort_values(['location', 'date']).reset_index(drop=True)

print(f"Rows after removing OWID aggregates : {df.shape[0]:,}")
print(f"Countries remaining                 : {df['location'].nunique()}")
print(f"Date column dtype                   : {df['date'].dtype}")


Rows after removing OWID aggregates : 395,311
Countries remaining                 : 237
Date column dtype                   : datetime64[ns]


### 8.2 Confirmed Cases

**`total_cases`** is a cumulative counter. Countries often miss reporting on specific days (weekends, public holidays), so the previous total remains valid during those gaps — forward-fill is appropriate. `cummax()` is applied afterward to correct any reported decreases, which occur when countries retroactively revise or reclassify confirmed cases.

**`new_cases`** is a daily flow. Negative values arise from retroactive corrections (a country subtracts previously over-counted cases from the current day's report) and are set to `NaN`. Missing days are left as `NaN` — a missing report is not the same as zero new cases.

**Smoothed and per-million variants** are derived metrics computed by OWID from the raw series. They are coerced to numeric and clipped at zero, but not re-filled. Back-filling a smoothed average would pull future values into earlier rows (temporal leakage).

**Note on `total_cases_per_million`:** this column is intentionally left null where `total_cases` is null. Filling it with zero would make the false claim that case count per million was zero, not simply unreported.


In [65]:
# --- total_cases (cumulative counter) ---
df['total_cases'] = pd.to_numeric(df['total_cases'], errors='coerce')
df['total_cases'] = df.groupby('location')['total_cases'].ffill()       # fill report gaps
df.loc[df['total_cases'] < 0, 'total_cases'] = np.nan                   # remove impossible values
df['total_cases'] = df.groupby('location')['total_cases'].cummax()      # fix retroactive dips
# Rows where total_cases remains null are early-pandemic rows with no reports yet.
# These rows are kept — dropping them would remove valid data in other columns.

# --- new_cases (daily flow) ---
df['new_cases'] = pd.to_numeric(df['new_cases'], errors='coerce')
df.loc[df['new_cases'] < 0, 'new_cases'] = np.nan   # retroactive corrections → NaN
# Do NOT substitute zero — missing report ≠ zero new cases

# --- new_cases_smoothed (7-day derived average) ---
df['new_cases_smoothed'] = pd.to_numeric(df['new_cases_smoothed'], errors='coerce')
df.loc[df['new_cases_smoothed'] < 0, 'new_cases_smoothed'] = np.nan
# Preserve nulls — back-filling would introduce temporal leakage

# --- per-million variants (derived; nulls preserved) ---
for col in ['total_cases_per_million', 'new_cases_per_million', 'new_cases_smoothed_per_million']:
    df[col] = pd.to_numeric(df[col], errors='coerce')
    df.loc[df[col] < 0, col] = np.nan
    df[col] = df[col].round(3)

print("Confirmed cases columns cleaned.")
print(df[['total_cases', 'new_cases', 'new_cases_smoothed']].describe().T)


Confirmed cases columns cleaned.
                          count           mean            std    min  \
total_cases        391,504.0000 1,828,868.3399 7,862,092.6062 0.0000   
new_cases          388,397.0000     1,997.0823    86,042.9852 0.0000   
new_cases_smoothed 387,232.0000     2,002.9862    32,517.2150 0.0000   

                          25%         50%          75%              max  
total_cases        5,507.0000 49,051.0000 562,705.0000 103,436,829.0000  
new_cases              0.0000      0.0000       0.0000  40,475,477.0000  
new_cases_smoothed     0.0000      8.7100     204.4300   5,782,211.0000  


### 8.3 Confirmed Deaths

The same logic as confirmed cases applies here. `total_deaths` is a cumulative counter (forward-fill within country, then `cummax()`). `new_deaths` is a daily flow (null out negatives, leave gaps as `NaN`). `new_deaths_smoothed` is a derived 7-day average — its nulls are preserved without additional filling.

Per-million variants are coerced to numeric, clipped at zero, and rounded. They are not independently re-filled.


In [66]:
# --- total_deaths (cumulative counter) ---
df['total_deaths'] = pd.to_numeric(df['total_deaths'], errors='coerce')
df['total_deaths'] = df.groupby('location')['total_deaths'].ffill()
df.loc[df['total_deaths'] < 0, 'total_deaths'] = np.nan
df['total_deaths'] = df.groupby('location')['total_deaths'].cummax()

# --- new_deaths (daily flow) ---
df['new_deaths'] = pd.to_numeric(df['new_deaths'], errors='coerce')
df.loc[df['new_deaths'] < 0, 'new_deaths'] = np.nan
# Do NOT substitute zero — missing report ≠ zero deaths

# --- new_deaths_smoothed (7-day derived average) ---
df['new_deaths_smoothed'] = pd.to_numeric(df['new_deaths_smoothed'], errors='coerce')
df.loc[df['new_deaths_smoothed'] < 0, 'new_deaths_smoothed'] = np.nan
# Preserve nulls

# --- per-million variants (derived; nulls preserved) ---
for col in ['total_deaths_per_million', 'new_deaths_per_million', 'new_deaths_smoothed_per_million']:
    df[col] = pd.to_numeric(df[col], errors='coerce')
    df.loc[df[col] < 0, col] = np.nan
    df[col] = df[col].round(3)

print("Confirmed deaths columns cleaned.")
print(df[['total_deaths', 'new_deaths', 'new_deaths_smoothed']].describe().T)


Confirmed deaths columns cleaned.
                           count        mean         std    min     25%  \
total_deaths        391,504.0000 20,467.3703 82,681.9527 0.0000 37.0000   
new_deaths          388,846.0000     18.1506    315.5595 0.0000  0.0000   
new_deaths_smoothed 387,681.0000     18.2033    118.2534 0.0000  0.0000   

                         50%        75%            max  
total_deaths        648.0000 7,395.0000 1,193,165.0000  
new_deaths            0.0000     0.0000    47,687.0000  
new_deaths_smoothed   0.0000     2.0000     6,812.4300  


### 8.4 Excess Mortality

Excess mortality quantifies deaths above the expected baseline for a given country and time period, regardless of attributed cause. It is widely considered a more complete indicator of pandemic mortality than confirmed COVID-19 deaths alone.

However, calculating excess mortality requires a reliable civil registration system with near-complete death reporting. This data exists for approximately 60 countries. For the remaining countries in this dataset, these columns are structurally null — the data does not exist, not merely unreported.

**Critically: these nulls must not be filled with zero.** A zero entry in `excess_mortality` is a real claim (mortality was at expected baseline). A null entry means the estimate cannot be made. Imputing zero here would systematically misrepresent the mortality experience of most low- and middle-income countries.

These metrics are also reported at weekly or monthly intervals, so most daily rows within a reporting period will be null by design.


In [67]:
excess_cols = [
    'excess_mortality',
    'excess_mortality_cumulative',
    'excess_mortality_cumulative_absolute',
    'excess_mortality_cumulative_per_million'
]

for col in excess_cols:
    df[col] = pd.to_numeric(df[col], errors='coerce')
    # Structural nulls are preserved — do NOT fill with zero

# Report coverage so analysts know what they are working with
coverage = df[excess_cols].notna().mean().mul(100).round(1)
print("Excess mortality coverage (% of rows with non-null values):")
print(coverage.to_string())


Excess mortality coverage (% of rows with non-null values):
excess_mortality                          3.4000
excess_mortality_cumulative               3.4000
excess_mortality_cumulative_absolute      3.4000
excess_mortality_cumulative_per_million   3.4000


### 8.5 Hospital & ICU Capacity

Hospital and ICU data was reported primarily by European countries and a small number of high-income countries elsewhere. Null rates across the full dataset will be high, and this is expected — not a cleaning failure.

**Daily point-in-time columns** (`icu_patients`, `hosp_patients` and their per-million variants) represent a census of current occupancy. Forward-fill is defensible for short reporting gaps (many countries reported every few days rather than daily). However, carrying a hospital census count forward over weeks of missing data produces fabricated figures. A fill limit of 3 days is applied.

**Weekly admission columns** (`weekly_icu_admissions`, `weekly_hosp_admissions`) represent totals over a 7-day window. These are forward-filled up to 7 days within each country, matching their reporting frequency. Analysts should treat these as weekly totals, not daily figures, when building visualizations.

**Prior approach and its problem:** An earlier version of this notebook forward-filled hospital columns without a limit, which silently extended the last observed census count through extended no-reporting periods — manufacturing data where none existed.


In [68]:
# Daily point-in-time columns: fill gaps up to 3 days
hosp_point_cols = [
    'icu_patients', 'icu_patients_per_million',
    'hosp_patients', 'hosp_patients_per_million'
]

for col in hosp_point_cols:
    df[col] = pd.to_numeric(df[col], errors='coerce')
    df.loc[df[col] < 0, col] = np.nan
    df[col] = df.groupby('location')[col].ffill(limit=3)  # 3-day cap on fill

# Weekly admission columns: fill gaps up to 7 days (matching reporting frequency)
hosp_weekly_cols = [
    'weekly_icu_admissions', 'weekly_icu_admissions_per_million',
    'weekly_hosp_admissions', 'weekly_hosp_admissions_per_million'
]

for col in hosp_weekly_cols:
    df[col] = pd.to_numeric(df[col], errors='coerce')
    df.loc[df[col] < 0, col] = np.nan
    df[col] = df.groupby('location')[col].ffill(limit=7)  # 7-day cap on fill

# Round per-million columns for storage efficiency
for col in ['hosp_patients_per_million', 'icu_patients_per_million',
            'weekly_icu_admissions_per_million', 'weekly_hosp_admissions_per_million']:
    df[col] = df[col].round(3)

print("Hospital and ICU columns cleaned.")


Hospital and ICU columns cleaned.


### 8.6 Stringency Index

The Oxford Stringency Index, produced by the Blavatnik School of Government, is a composite measure (0–100) of government policy responses — including school closures, workplace restrictions, travel bans, and stay-at-home orders. A higher score indicates more restrictive policies.

The index updates when policies change, not on every calendar day. Forward-fill within each country is therefore the correct treatment: a country's policies remain in effect until explicitly changed, and a missing daily row reflects a reporting gap in the index, not a policy change.

Values outside [0, 100] cannot represent a valid policy score and are treated as encoding errors.


In [69]:
df['stringency_index'] = pd.to_numeric(df['stringency_index'], errors='coerce')

# Values outside the valid [0, 100] range are encoding errors
df.loc[(df['stringency_index'] < 0) | (df['stringency_index'] > 100), 'stringency_index'] = np.nan

# Forward-fill within country — policies remain in effect until changed
df['stringency_index'] = df.groupby('location')['stringency_index'].ffill()

print(df['stringency_index'].describe())


count   307,467.0000
mean         32.4103
std          24.9731
min           0.0000
25%          11.1100
50%          25.7400
75%          50.9300
max         100.0000
Name: stringency_index, dtype: float64


### 8.7 Reproduction Rate

The effective reproduction number R(t) is a modeled estimate of how many secondary infections each infected person generates on average at a given point in time. A value above 1.0 indicates exponential growth; below 1.0 indicates a declining epidemic.

R(t) is not a direct measurement — it is derived from observed case trends using epidemiological models with uncertainty ranges. The single-column value here does not capture that uncertainty.

Negative values are physically impossible and are treated as encoding errors. Forward-fill is applied with a 7-day limit: R(t) can change quickly during active outbreaks, so carrying a stale estimate beyond one week risks misrepresenting the epidemic trajectory.


In [70]:
df['reproduction_rate'] = pd.to_numeric(df['reproduction_rate'], errors='coerce')

# Negative R(t) is impossible — encoding error
df.loc[df['reproduction_rate'] < 0, 'reproduction_rate'] = np.nan

# Limited forward-fill (7 days): R(t) can shift quickly; stale estimates beyond a week are misleading
df['reproduction_rate'] = df.groupby('location')['reproduction_rate'].ffill(limit=7)

print(df['reproduction_rate'].describe())


count   184,717.0000
mean          0.9070
std           0.4012
min           0.0000
25%           0.7100
50%           0.9500
75%           1.1400
max           5.8700
Name: reproduction_rate, dtype: float64


### 8.8 Testing Columns

Testing data requires careful handling because countries used different definitions of "a test," reported at inconsistent frequencies, and improved their testing infrastructure at different rates over the pandemic.

**`total_tests`** — cumulative counter. Same strategy as `total_cases`: forward-fill within country, then `cummax()`.

**`new_tests`** — daily flow. Negatives are set to `NaN`; gaps remain `NaN`. Do not substitute zero.

**`new_tests_smoothed`** — 7-day derived average. Nulls preserved; not back-filled.

**`positive_rate`** — proportion bounded [0, 1]. Values outside this range are encoding errors. Forward-filled with a 7-day limit, since this is a rolling average that changes gradually.

**`tests_per_case`** — the mathematical inverse of `positive_rate`. Very large values (including `Inf`) arise when new cases approach zero in the denominator. These are nulled.

**`tests_units`** — categorical string describing what the country counted as a "test" (people tested, tests performed, or samples tested). Countries using different definitions cannot be directly compared on testing volume. The category is normalized and forward-filled within country.

**Note on `new_tests_per_thousand`:** this column is used as-is from OWID's pre-computed values. An earlier approach recomputed it as `new_tests / population * 1000` and then forward-filled the result. That approach overwrites OWID's original values with a recomputed version derived from already-cleaned `new_tests`, compounding earlier modifications. OWID's column is more reliable.


In [71]:
# --- total_tests (cumulative counter) ---
df['total_tests'] = pd.to_numeric(df['total_tests'], errors='coerce')
df['total_tests'] = df.groupby('location')['total_tests'].ffill()
df.loc[df['total_tests'] < 0, 'total_tests'] = np.nan
df['total_tests'] = df.groupby('location')['total_tests'].cummax()

# --- total_tests_per_thousand (cumulative, derived) ---
df['total_tests_per_thousand'] = pd.to_numeric(df['total_tests_per_thousand'], errors='coerce')
df['total_tests_per_thousand'] = df.groupby('location')['total_tests_per_thousand'].ffill()
df['total_tests_per_thousand'] = df.groupby('location')['total_tests_per_thousand'].cummax()
df.loc[df['total_tests_per_thousand'] < 0, 'total_tests_per_thousand'] = np.nan
df['total_tests_per_thousand'] = df['total_tests_per_thousand'].round(3)

# --- new_tests (daily flow) ---
df['new_tests'] = pd.to_numeric(df['new_tests'], errors='coerce')
df.loc[df['new_tests'] < 0, 'new_tests'] = np.nan
# Do NOT substitute zero — missing report ≠ zero tests

# --- new_tests_per_thousand (use OWID's pre-computed column; do not recompute) ---
df['new_tests_per_thousand'] = pd.to_numeric(df['new_tests_per_thousand'], errors='coerce')
df.loc[df['new_tests_per_thousand'] < 0, 'new_tests_per_thousand'] = np.nan
df['new_tests_per_thousand'] = df['new_tests_per_thousand'].round(3)

# --- new_tests_smoothed (7-day derived average) ---
df['new_tests_smoothed'] = pd.to_numeric(df['new_tests_smoothed'], errors='coerce')
df.loc[df['new_tests_smoothed'] < 0, 'new_tests_smoothed'] = np.nan
# Preserve nulls — do NOT back-fill

# --- new_tests_smoothed_per_thousand ---
df['new_tests_smoothed_per_thousand'] = pd.to_numeric(df['new_tests_smoothed_per_thousand'], errors='coerce')
df.loc[df['new_tests_smoothed_per_thousand'] < 0, 'new_tests_smoothed_per_thousand'] = np.nan
df['new_tests_smoothed_per_thousand'] = df['new_tests_smoothed_per_thousand'].round(3)

# --- positive_rate (proportion in [0, 1]) ---
df['positive_rate'] = pd.to_numeric(df['positive_rate'], errors='coerce')
df.loc[(df['positive_rate'] < 0) | (df['positive_rate'] > 1), 'positive_rate'] = np.nan
df['positive_rate'] = df.groupby('location')['positive_rate'].ffill(limit=7)

# --- tests_per_case (inverse of positive_rate; Inf occurs when new_cases → 0) ---
df['tests_per_case'] = pd.to_numeric(df['tests_per_case'], errors='coerce')
df.loc[df['tests_per_case'] < 0, 'tests_per_case'] = np.nan
df.loc[df['tests_per_case'] == np.inf, 'tests_per_case'] = np.nan
df['tests_per_case'] = df.groupby('location')['tests_per_case'].ffill(limit=7)

# --- tests_units (categorical string) ---
valid_units = {'people tested', 'tests performed', 'samples tested'}
df['tests_units'] = df['tests_units'].astype(str).str.strip().str.lower()
df['tests_units'] = df['tests_units'].where(df['tests_units'].isin(valid_units), other=np.nan)
df['tests_units'] = df.groupby('location')['tests_units'].ffill()
df['tests_units'] = df['tests_units'].astype('category')

print("Testing columns cleaned.")
print(f"tests_units categories : {df['tests_units'].cat.categories.tolist()}")


Testing columns cleaned.
tests_units categories : ['people tested', 'samples tested', 'tests performed']


### 8.9 Vaccination Columns

**Cumulative counters** (`total_vaccinations`, `people_vaccinated`, `people_fully_vaccinated`, `total_boosters`) follow the same strategy as confirmed cases: forward-fill within country to handle reporting gaps (vaccination data is typically reported every few days, not daily), then `cummax()` to correct downward revisions.

**`new_vaccinations`** is a daily flow. Negatives are set to `NaN`; gaps remain `NaN`. Critically, this column is not clipped at a quantile ceiling. Earlier implementations applied a 99th-percentile cap to "remove outliers," which distorts real surge data: India and China, for instance, administered hundreds of millions of doses in single-day reporting events. Clipping those values would understate actual vaccination rates.

**Smoothed and per-hundred/per-million variants** are derived metrics. Nulls are preserved.

**Note on `new_vaccinations_smoothed`:** an earlier version applied `fillna(0)` to this column before converting it to numeric. That operation silently converted all non-numeric strings (including truly missing entries stored as empty strings) to zero — a latent bug corrected here by converting to numeric first.


In [72]:
# --- cumulative vaccination counters ---
cum_vax_cols = [
    'total_vaccinations',
    'people_vaccinated',
    'people_fully_vaccinated',
    'total_boosters'
]

for col in cum_vax_cols:
    df[col] = pd.to_numeric(df[col], errors='coerce')
    df[col] = df.groupby('location')[col].ffill()
    df.loc[df[col] < 0, col] = np.nan
    df[col] = df.groupby('location')[col].cummax()

# --- new_vaccinations (daily flow) ---
df['new_vaccinations'] = pd.to_numeric(df['new_vaccinations'], errors='coerce')
df.loc[df['new_vaccinations'] < 0, 'new_vaccinations'] = np.nan
# Do NOT clip at a quantile — surge data for large countries is real, not an outlier
# Do NOT fill with zero — missing report ≠ zero vaccinations

# --- new_vaccinations_smoothed (7-day derived average) ---
df['new_vaccinations_smoothed'] = pd.to_numeric(df['new_vaccinations_smoothed'], errors='coerce')
# Preserve nulls — do NOT fill with zero

# --- cumulative per-hundred variants ---
per_hundred_cols = [
    'total_vaccinations_per_hundred',
    'people_vaccinated_per_hundred',
    'people_fully_vaccinated_per_hundred',
    'total_boosters_per_hundred'
]
for col in per_hundred_cols:
    df[col] = pd.to_numeric(df[col], errors='coerce')
    df[col] = df.groupby('location')[col].ffill()
    df.loc[df[col] < 0, col] = np.nan
    df[col] = df[col].round(3)

# --- smoothed flow variants ---
smoothed_vax_cols = [
    'new_vaccinations_smoothed_per_million',
    'new_people_vaccinated_smoothed',
    'new_people_vaccinated_smoothed_per_hundred'
]
for col in smoothed_vax_cols:
    df[col] = pd.to_numeric(df[col], errors='coerce')
    df.loc[df[col] < 0, col] = np.nan
    # Preserve nulls — derived columns

print("Vaccination columns cleaned.")
print(df[cum_vax_cols].describe().T)


Vaccination columns cleaned.
                               count            mean              std    min  \
total_vaccinations      273,878.0000 49,309,699.0247 252,710,670.9940 0.0000   
people_vaccinated       273,192.0000 21,507,583.1432 105,482,121.5036 0.0000   
people_fully_vaccinated 261,093.0000 19,922,571.0955 100,575,193.6326 1.0000   
total_boosters          190,064.0000 12,508,450.8302  59,915,092.0371 1.0000   

                                 25%            50%             75%  \
total_vaccinations      340,020.7500 3,179,647.0000 18,808,671.0000   
people_vaccinated       188,475.0000 1,991,775.5000  8,476,785.7500   
people_fully_vaccinated 176,005.0000 1,631,699.0000  7,691,954.0000   
total_boosters           49,213.0000   657,967.0000  5,200,841.0000   

                                       max  
total_vaccinations      3,491,077,000.0000  
people_vaccinated       1,310,292,000.0000  
people_fully_vaccinated 1,276,760,000.0000  
total_boosters            826,913,

### 8.10 Population & Static Country Indicators

These columns carry a single fixed value per country — the same number appears on every row for that country, or the column is null throughout. Examples include population, GDP per capita, median age, hospital beds per thousand, and life expectancy.

The correct treatment is to convert to numeric, then forward-fill and back-fill within each country. This propagates the country's value to all rows, regardless of where in the time series the non-null entry happened to appear.

Two important constraints:
- No cross-country imputation. A missing value for one country cannot be inferred from another's demographic data.
- Some indicators — notably `extreme_poverty`, `female_smokers`, and `handwashing_facilities` — are genuinely absent for many countries in international statistics databases. These structural gaps remain null after filling.

**Note on prior blanket-fill approach:** an earlier version applied `groupby('location').apply(lambda x: x.ffill().bfill())` across the entire DataFrame at the end of the notebook. This is problematic for two reasons. First, it processes time-varying columns alongside static ones, silently back-filling daily case counts using values from later rows. Second, it provides no column-type awareness, treating a smoothed average the same as a demographic indicator. The approach in this notebook handles each column group explicitly.


In [73]:
# --- population ---
df['population'] = pd.to_numeric(df['population'], errors='coerce')
df.loc[df['population'] <= 0, 'population'] = np.nan            # zero or negative population is invalid
df['population'] = df.groupby('location')['population'].ffill().bfill()

# --- population density ---
df['population_density'] = pd.to_numeric(df['population_density'], errors='coerce')
df['population_density'] = df.groupby('location')['population_density'].ffill().bfill()

# --- demographic indicators ---
demo_cols = ['median_age', 'aged_65_older', 'aged_70_older']
for col in demo_cols:
    df[col] = pd.to_numeric(df[col], errors='coerce')
    df[col] = df.groupby('location')[col].ffill().bfill()

# --- economic indicators ---
econ_cols = ['gdp_per_capita', 'extreme_poverty']
for col in econ_cols:
    df[col] = pd.to_numeric(df[col], errors='coerce')
    df[col] = df.groupby('location')[col].ffill().bfill()

# --- health system indicators ---
health_cols = [
    'cardiovasc_death_rate', 'diabetes_prevalence',
    'female_smokers', 'male_smokers',
    'handwashing_facilities', 'hospital_beds_per_thousand',
    'life_expectancy', 'human_development_index'
]
for col in health_cols:
    df[col] = pd.to_numeric(df[col], errors='coerce')
    df[col] = df.groupby('location')[col].ffill().bfill()

print("Static country indicators cleaned.")

# Remaining nulls after fill represent genuine data gaps in international statistics
coverage_check = df[['population'] + demo_cols + econ_cols + health_cols].isnull().mean().mul(100).round(1)
print("\nResidual null % (structural gaps only, after fill):")
print(coverage_check.to_string())


Static country indicators cleaned.

Residual null % (structural gaps only, after fill):
population                   0.0000
median_age                   0.0000
aged_65_older                0.0000
aged_70_older                0.0000
gdp_per_capita               0.0000
extreme_poverty              0.0000
cardiovasc_death_rate        0.0000
diabetes_prevalence          0.0000
female_smokers               0.0000
male_smokers                 0.0000
handwashing_facilities       0.0000
hospital_beds_per_thousand   0.0000
life_expectancy              0.0000
human_development_index      0.0000


## 9. Post-Cleaning Validation

Before exporting, this section verifies that the cleaning operations produced the expected results. Three checks are run:

1. **Monotonicity of cumulative counters** — confirms that no country has a cumulative column that decreases after cleaning.
2. **Absence of impossible values** — confirms that no negative values remain in columns that were explicitly cleaned.
3. **Static indicator consistency** — confirms that population and demographic columns now have a consistent value per country (within-country variance = 0 after filling).


In [74]:
# ── Check 1: Cumulative columns are non-decreasing within each country ──────────────
cumulative_cols = [
    'total_cases', 'total_deaths', 'total_tests',
    'total_vaccinations', 'people_vaccinated', 'people_fully_vaccinated', 'total_boosters'
]

violations = {}
for col in cumulative_cols:
    if col not in df.columns:
        continue
    decreasing = (
        df.dropna(subset=[col])
          .groupby('location')[col]
          .apply(lambda s: (s.diff() < 0).sum())
    )
    n_violations = (decreasing > 0).sum()
    violations[col] = n_violations

print("Monotonicity violations remaining after cleaning (should be 0):")
for col, n in violations.items():
    status = "OK" if n == 0 else f"ISSUE — {n} countries"
    print(f"  {col:<45} {status}")


Monotonicity violations remaining after cleaning (should be 0):
  total_cases                                   OK
  total_deaths                                  OK
  total_tests                                   OK
  total_vaccinations                            OK
  people_vaccinated                             OK
  people_fully_vaccinated                       OK
  total_boosters                                OK


In [75]:
# ── Check 2: No negative values in cleaned flow columns ──────────────────────────────
flow_cols = ['new_cases', 'new_deaths', 'new_tests', 'new_vaccinations']

print("Negative value count in daily flow columns (should be 0):")
for col in flow_cols:
    if col not in df.columns:
        continue
    n_neg = (df[col] < 0).sum()
    status = "OK" if n_neg == 0 else f"ISSUE — {n_neg} rows"
    print(f"  {col:<45} {status}")


Negative value count in daily flow columns (should be 0):
  new_cases                                     OK
  new_deaths                                    OK
  new_tests                                     OK
  new_vaccinations                              OK


In [76]:
# ── Check 3: Static indicators are internally consistent per country ──────────────────
static_check_cols = ['population', 'median_age', 'gdp_per_capita', 'life_expectancy']

print("Within-country variance for static indicators (should be 0 or NaN):")
for col in static_check_cols:
    if col not in df.columns:
        continue
    max_var = df.groupby('location')[col].var().max()
    status = "OK" if (pd.isna(max_var) or max_var == 0) else f"ISSUE — max variance: {max_var:.4f}"
    print(f"  {col:<45} {status}")


Within-country variance for static indicators (should be 0 or NaN):
  population                                    OK
  median_age                                    OK
  gdp_per_capita                                OK
  life_expectancy                               OK


## 10. Cleaned Dataset Summary


In [77]:
print("=== Final Dataset Shape ===")
print(f"Rows    : {df.shape[0]:,}")
print(f"Columns : {df.shape[1]}")

print("\n=== Date Range ===")
print(f"{df['date'].min().date()}  →  {df['date'].max().date()}")

print("\n=== Country Count ===")
print(f"Unique countries : {df['location'].nunique()}")

print("\n=== Residual Null % — Top 20 Columns ===")
remaining_null = df.isnull().mean().mul(100).round(1).sort_values(ascending=False).head(20)
print(remaining_null.to_string())


=== Final Dataset Shape ===
Rows    : 395,311
Columns : 66

=== Date Range ===
2020-01-01  →  2024-08-14

=== Country Count ===


Unique countries : 237

=== Residual Null % — Top 20 Columns ===
excess_mortality_cumulative_per_million      96.6000
excess_mortality                             96.6000
excess_mortality_cumulative                  96.6000
excess_mortality_cumulative_absolute         96.6000
weekly_icu_admissions_per_million            94.3000
weekly_icu_admissions                        94.3000
weekly_hosp_admissions_per_million           91.6000
weekly_hosp_admissions                       91.6000
hosp_patients_per_million                    90.2000
hosp_patients                                90.2000
icu_patients                                 90.1000
icu_patients_per_million                     90.1000
new_vaccinations                             86.8000
new_tests_per_thousand                       81.0000
new_tests                                    81.0000
tests_per_case                               75.2000
positive_rate                                75.0000
new_tests_smoothed_per_thousand   

In [78]:
# Final column types — verify that no columns remain as generic 'object' unexpectedly
df.dtypes.to_frame('dtype').reset_index().rename(columns={'index': 'column'})


,column,dtype
0,continent,object
1,location,object
2,date,datetime64[ns]
3,total_cases,float64
4,new_cases,float64
...,...,...
61,population,float64
62,excess_mortality_cumulative_absolute,float64
63,excess_mortality_cumulative,float64
64,excess_mortality,float64


## 11. Export Cleaned Dataset


In [79]:
OUTPUT_PATH = r'F:\faculty\Level 3 S_2\Data Visualization\Global COVID-19 Pandemic Analysis\Global-COVID-19-Pandemic-Analysis\Data\Processed\owid_covid_cleaned.csv'
df.to_csv(OUTPUT_PATH, index=False)
print(f"Cleaned dataset saved to  : {OUTPUT_PATH}")
print(f"Shape                     : {df.shape[0]:,} rows × {df.shape[1]} columns")


Cleaned dataset saved to  : F:\faculty\Level 3 S_2\Data Visualization\Global COVID-19 Pandemic Analysis\Global-COVID-19-Pandemic-Analysis\Data\Processed\owid_covid_cleaned.csv
Shape                     : 395,311 rows × 66 columns
